# 1. Install & Import

In [ ]:
!pip install datasets sacrebleu -q

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

import numpy as np
import re, random, time, math
from collections import Counter
import sacrebleu
from datasets import load_dataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

## 2. Load Dataset — Tatoeba (ar → en)

> **Why Tatoeba?**  
> opus_books is literary text — long, complex, varied. Tatoeba contains short everyday sentences
> ("Where is the station?", "I like coffee") that are much easier for an LSTM to learn from.
> This alone will significantly reduce your loss.

In [ ]:
from datasets import load_dataset

# Fallback: opus100 ar-en (standard Parquet format, no script needed)
dataset = load_dataset('Helsinki-NLP/opus-100', 'ar-en')
print(dataset)

all_data = []
for split in dataset:
    for ex in dataset[split]:
        all_data.append((ex['translation']['ar'], ex['translation']['en']))

print(f'\nTotal pairs: {len(all_data)}')
print('Sample:', all_data[0])

In [ ]:
# Keep only 80k pairs — enough to learn, fast enough to train
random.shuffle(all_data)
all_data = all_data[:80_000]
print(f'Using: {len(all_data)} pairs')